In [ ]:
import torch
from IPython.display import HTML, display

if not torch.cuda.is_available():
    raise SystemExit("⚠️ GPU Not Detected! Enable GPU in Settings → Accelerator.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
cuda_ver = torch.version.cuda

display(HTML(f"""
<div style='background-color:#e8f5e9;padding:15px;border-radius:8px;border-left:5px solid #4caf50;'>
  <h3 style='color:#2e7d32;margin:0 0 8px 0;'>✅ Hardware Accelerator Ready</h3>
  <table style='font-family:monospace;font-size:13px;'>
    <tr><td><b>GPU Device:</b></td><td>&nbsp;{gpu_name}</td></tr>
    <tr><td><b>Total VRAM:</b></td><td>&nbsp;{vram_gb} GB</td></tr>
    <tr><td><b>CUDA Version:</b></td><td>&nbsp;{cuda_ver}</td></tr>
    <tr><td><b>PyTorch:</b></td><td>&nbsp;{torch.__version__}</td></tr>
  </table>
</div>
"""))

In [ ]:
import os
from IPython.display import HTML, display

print("🚀 Setting up ComfyUI & YuE2 environment...")
os.chdir('/kaggle/working')

if not os.path.exists("/kaggle/working/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /kaggle/working/ComfyUI

%cd /kaggle/working/ComfyUI

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q soundfile torchaudio accelerate sentencepiece mutagen websocket-client safetensors ipywidgets librosa
!apt-get -y install -qq aria2 ffmpeg > /dev/null 2>&1

display(HTML("<div style='color:green;'><b>✅ Environment installed!</b></div>"))

In [ ]:
# ============================================================
# 2.5. Install Custom Nodes for YuE2
# ============================================================
import os

%cd /kaggle/working/ComfyUI/custom_nodes

# Клонируем ноды для YuE2 (если они есть в публичном репозитории)
# Примечание: если репозитория нет, ноды уже встроены в Comfy-Org версию

print("📦 Checking for YuE2 custom nodes...")

# Проверяем, установлены ли уже ноды
if not os.path.exists("ComfyUI-YuE2"):
    print("⚠️ Custom nodes for YuE2 not found. Will try to use built-in nodes.")
    print("💡 If generation fails, you may need to manually install YuE2 nodes from:")
    print("   https://github.com/comfyanonymous/ComfyUI (check if YuE2 support is built-in)")
else:
    print("✅ Custom nodes found.")

%cd /kaggle/working/ComfyUI

In [ ]:
import os
import time
import subprocess
import urllib.request
from IPython.display import HTML, display

%cd /kaggle/working/ComfyUI

model_dir = "/kaggle/working/ComfyUI/models/checkpoints"
encoder_dir = "/kaggle/working/ComfyUI/models/audio_encoders"
os.makedirs(model_dir, exist_ok=True)
os.makedirs(encoder_dir, exist_ok=True)

model_path = os.path.join(model_dir, "yue2_3b_int8_convrot.safetensors")
encoder_path = os.path.join(encoder_dir, "sheetsage2_bf16.safetensors")

if not os.path.exists(model_path) or os.path.getsize(model_path) < 4_000_000_000:
    print("📥 Downloading YuE2-3B INT8 (~4.5 GB)...")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
      "https://huggingface.co/Comfy-Org/YuE2/resolve/main/checkpoints/yue2_3b_int8_convrot.safetensors" \
      -d "{model_dir}" -o "yue2_3b_int8_convrot.safetensors"
else:
    print("⚡ INT8 checkpoint already in cache.")

if not os.path.exists(encoder_path) or os.path.getsize(encoder_path) < 1_000_000_000:
    print("📥 Downloading SheetSage2 Encoder (~1.8 GB)...")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
      "https://huggingface.co/Comfy-Org/YuE2/resolve/main/audio_encoders/sheetsage2_bf16.safetensors" \
      -d "{encoder_dir}" -o "sheetsage2_bf16.safetensors"
else:
    print("⚡ SheetSage2 encoder already in cache.")

!pkill -9 -f "main.py" || True
time.sleep(2)

server_log = open("/kaggle/working/comfy_server.log", "w")
server_cmd = ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188", "--highvram", "--dont-upcast-attention"]
proc = subprocess.Popen(server_cmd, stdout=server_log, stderr=server_log, cwd="/kaggle/working/ComfyUI")

server_alive = False
for i in range(40):
    if proc.poll() is not None: break
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        server_alive = True
        break
    except Exception:
        time.sleep(1.5)

if server_alive:
    display(HTML("<div style='background-color:#e8f5e9;padding:15px;border-radius:8px;border-left:5px solid #4caf50;'><b style='color:#2e7d32;'>🟢 YuE2-3B INT8 Engine Online!</b><br><span style='color:#555;'>Run Block 4.</span></div>"))
else:
    print("❌ Startup failed. Log (/kaggle/working/comfy_server.log):")
    with open("/kaggle/working/comfy_server.log", "r") as f:
        print(f.read())

In [ ]:
import gradio as gr
import json, time, random, glob, os, uuid, shutil, websocket, urllib.request

ckpt_name = "yue2_3b_int8_convrot.safetensors"

preset_dict = {
    "Indie Pop Anthem": "Upbeat indie pop with warm female vocals, bright electric guitars, punchy drums, melodic bass, and subtle synth layers. Catchy and energetic, with an uplifting summer atmosphere, a memorable chorus, and polished modern production.",
    "Synthwave 80s Retro": "1980s retro synthwave, analog vintage synthesizers, gated snare reverb, driving arpeggiated bassline, male nostalgic vocals, neon aesthetic, cinematic nocturnal drive.",
    "Lo-Fi Chill Hop": "Warm lofi hip hop, vinyl crackle, mellow electric piano chords, smooth relaxed boom bap drums, deep warm sub bass, peaceful afternoon coffee shop atmosphere.",
    "Cinematic Epic Orchestral": "Epic cinematic trailer music, soaring brass ensemble, thundering Taiko drums, expressive emotional strings, dramatic crescendos, heroic orchestral climax.",
    "Modern Alt Rock": "Modern alternative rock, overdriven crunchy guitars, aggressive live acoustic drum kit, energetic lead vocals, heavy bassline, anthemic chorus."
}

def build_workflow(style_text, lyrics_text, comp_mode, enable_abc, sampler, scheduler, steps, fmt, quality, prefix, seed):
    return {
      "10": {"inputs": {"filename_prefix": prefix, "format": fmt, "format.quality": quality, "audio": ["33:9", 0]}, "class_type": "SaveAudioAdvanced"},
      "33:15": {"inputs": {"ckpt_name": ckpt_name}, "class_type": "CheckpointLoaderSimple"},
      "33:24": {"inputs": {"style": ["33:36", 0], "lyrics": ["33:37", 0], "seed": ["33:34", 0], "mode": comp_mode, "max_abc_tokens": 8192, "temperature": 0.7, "top_p": 0.9, "top_k": 30, "repetition_penalty": 1.005, "penalty_window": 100, "clip": ["33:15", 1]}, "class_type": "YuE2GenerateABC"},
      "33:14": {"inputs": {"source": ["33:30", 0]}, "class_type": "PreviewAny"},
      "33:30": {"inputs": {"switch": ["33:31", 0], "on_false": ["33:32", 0], "on_true": ["33:24", 0]}, "class_type": "ComfySwitchNode"},
      "33:32": {"inputs": {"value": ""}, "class_type": "PrimitiveString"},
      "33:25": {"inputs": {"style": ["33:36", 0], "lyrics": ["33:37", 0], "abc": ["33:14", 0], "seed": ["33:34", 0], "mode": comp_mode, "max_duration": 360, "temperature": 1.0, "top_p": 0.95, "top_k": 100, "repetition_penalty": 1.2, "clip": ["33:15", 1]}, "class_type": "YuE2GenerateMusic"},
      "33:31": {"inputs": {"value": enable_abc}, "class_type": "PrimitiveBoolean"},
      "33:18": {"inputs": {"conditioning": ["33:25", 0]}, "class_type": "ConditioningZeroOut"},
      "33:23": {"inputs": {"source": ["33:25", 1]}, "class_type": "PreviewAny"},
      "33:8": {"inputs": {"seed": ["33:34", 0], "steps": steps, "cfg": 1.0, "sampler_name": sampler, "scheduler": scheduler, "denoise": 1.0, "model": ["33:15", 0], "positive": ["33:25", 0], "negative": ["33:18", 0], "latent_image": ["33:5", 0]}, "class_type": "KSampler"},
      "33:5": {"inputs": {"seconds": ["33:25", 1], "batch_size": 1}, "class_type": "EmptyYuE2LatentAudio"},
      "33:9": {"inputs": {"samples": ["33:8", 0], "vae": ["33:15", 2]}, "class_type": "VAEDecodeAudio"},
      "33:34": {"inputs": {"seed": seed}, "class_type": "SeedNode"},
      "33:36": {"inputs": {"value": style_text}, "class_type": "PrimitiveStringMultiline"},
      "33:37": {"inputs": {"value": lyrics_text}, "class_type": "PrimitiveStringMultiline"}
    }

def run_comfy(workflow_api, steps, progress):
    client_id = str(uuid.uuid4())
    ws = websocket.WebSocket()
    ws.connect("ws://127.0.0.1:8188/ws?clientId=" + client_id)
    req_payload = json.dumps({"prompt": workflow_api, "client_id": client_id}).encode('utf-8')
    req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=req_payload, headers={'Content-Type': 'application/json'})
    resp = urllib.request.urlopen(req)
    prompt_id = json.loads(resp.read())['prompt_id']
    start = time.time()
    err = None
    while True:
        msg = ws.recv()
        if isinstance(msg, str):
            event = json.loads(msg)
            et = event.get('type')
            ed = event.get('data', {})
            if et == 'execution_error':
                err = ed.get('exception_message', 'Unknown error')
                break
            if et == 'executing':
                nid = ed.get('node')
                if nid is None:
                    if ed.get('prompt_id') == prompt_id:
                        break
            elif et == 'progress':
                cur = ed.get('value', 0)
                mx = ed.get('max', steps)
                if mx:
                    progress(0.3 + (cur / mx) * 0.65, desc="Sampling {}/{}".format(cur, mx))
    ws.close()
    return err, round(time.time() - start, 1)

def find_output(prefix, fmt):
    files = glob.glob("/kaggle/working/ComfyUI/output/" + prefix + "*." + fmt)
    if not files:
        return None
    return max(files, key=os.path.getctime)

def generate_new_song(genre, custom_style, lyrics, enable_abc, sampler, scheduler, steps, fmt, quality, progress=gr.Progress()):
    if not lyrics or not lyrics.strip():
        return None, "❌ Lyrics required!"
    progress(0.05, desc="Preparing...")
    style_text = custom_style if genre == "Custom" else preset_dict.get(genre, custom_style)
    seed = random.randint(1, 2147483647)
    prefix = "audio/YuE2G_" + str(int(time.time()))
    wf = build_workflow(style_text, lyrics, 'full', enable_abc, sampler, scheduler, int(steps), fmt, quality, prefix, seed)
    err, elapsed = run_comfy(wf, int(steps), progress)
    if err:
        return None, "❌ " + str(err)
    latest = find_output(prefix, fmt)
    if not latest:
        return None, "❌ No output file found."
    progress(1.0, desc="Done!")
    return latest, "✅ Done in {}s | Seed: {} | Mode: New Song".format(elapsed, seed)

def generate_cover(source_audio, target_style, target_lyrics, sampler, scheduler, steps, fmt, quality, progress=gr.Progress()):
    if not source_audio:
        return None, "❌ Upload source audio first."
    progress(0.05, desc="Preparing cover...")
    ref = "/tmp/cover_ref_" + str(int(time.time())) + "_" + os.path.basename(source_audio)
    shutil.copy(source_audio, ref)
    lyrics_text = target_lyrics if (target_lyrics and target_lyrics.strip()) else "[Instrumental]"
    seed = random.randint(1, 2147483647)
    prefix = "audio/YuE2GC_" + str(int(time.time()))
    wf = build_workflow(target_style, lyrics_text, 'melody', True, sampler, scheduler, int(steps), fmt, quality, prefix, seed)
    err, elapsed = run_comfy(wf, int(steps), progress)
    if err:
        return None, "❌ " + str(err)
    latest = find_output(prefix, fmt)
    if not latest:
        return None, "❌ No output file found."
    progress(1.0, desc="Done!")
    return latest, "✅ Cover done in {}s | Seed: {} | Ref: {}".format(elapsed, seed, os.path.basename(ref))

with gr.Blocks(title="YuE2 Music Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎵 YuE2 Music Studio (Web)\n**Model:** YuE2-3B INT8 (latest, Comfy-Org) • Optimized for Kaggle T4")

    with gr.Tabs():
        with gr.Tab("🎵 New Song"):
            with gr.Row():
                with gr.Column():
                    n_genre = gr.Dropdown(["Custom"] + list(preset_dict.keys()), value="Indie Pop Anthem", label="Genre Preset")
                    n_style = gr.Textbox(value=preset_dict["Indie Pop Anthem"], label="Style Description", lines=3)
                    n_lyrics = gr.Textbox(value="[Verse]\nMorning light across the window\nCity waking down below\n\n[Chorus]\nRun with me into the sunlight\nLeave the shadows far behind\n\n[Outro]", label="Lyrics ([Verse]/[Chorus]/[Outro])", lines=10)
                with gr.Column():
                    n_abc = gr.Checkbox(False, label="Enable ABC Planning (slower, better structure)")
                    n_samp = gr.Dropdown(["dpm_2","euler","dpmpp_2m","heun","dpmpp_sde"], value="dpm_2", label="Sampler")
                    n_sched = gr.Dropdown(["sgm_uniform","normal","karras","simple"], value="sgm_uniform", label="Scheduler")
                    n_steps = gr.Slider(10, 50, 32, step=1, label="Steps")
                    n_fmt = gr.Radio(["mp3","wav","flac"], value="mp3", label="Format")
                    n_qual = gr.Dropdown(["320k","256k","192k","128k"], value="320k", label="Quality")
            n_btn = gr.Button("🎵 Generate New Song", variant="primary", size="lg")
            n_audio = gr.Audio(label="Result", type="filepath")
            n_status = gr.Textbox(label="Status", interactive=False)

        with gr.Tab("🔄 Cover / Restyle"):
            with gr.Row():
                with gr.Column():
                    c_src = gr.Audio(label="Source Audio (reference upload)", type="filepath")
                    c_style = gr.Textbox(value="Jazz-funk cover, smooth saxophone, upright bass", label="Target Style", lines=3)
                    c_lyrics = gr.Textbox(value="", label="Lyrics (optional, empty = instrumental)", lines=6)
                with gr.Column():
                    c_samp = gr.Dropdown(["dpm_2","euler","dpmpp_2m","heun","dpmpp_sde"], value="dpm_2", label="Sampler")
                    c_sched = gr.Dropdown(["sgm_uniform","normal","karras","simple"], value="sgm_uniform", label="Scheduler")
                    c_steps = gr.Slider(10, 50, 32, step=1, label="Steps")
                    c_fmt = gr.Radio(["mp3","wav","flac"], value="mp3", label="Format")
                    c_qual = gr.Dropdown(["320k","256k","192k","128k"], value="320k", label="Quality")
            c_btn = gr.Button("🔄 Generate Cover", variant="primary", size="lg")
            c_audio = gr.Audio(label="Result", type="filepath")
            c_status = gr.Textbox(label="Status", interactive=False)

    def upd_style(g):
        return preset_dict.get(g, "") if g != "Custom" else gr.update()
    n_genre.change(upd_style, n_genre, n_style)

    n_btn.click(generate_new_song, [n_genre, n_style, n_lyrics, n_abc, n_samp, n_sched, n_steps, n_fmt, n_qual], [n_audio, n_status])
    c_btn.click(generate_cover, [c_src, c_style, c_lyrics, c_samp, c_sched, c_steps, c_fmt, c_qual], [c_audio, c_status])

demo.launch(share=True, debug=False)

In [ ]:
import glob
import os
import base64
import zipfile
import time
from IPython.display import display, Javascript, HTML, FileLink

def smart_download_tracks():
    search_paths = [
        "/kaggle/working/ComfyUI/output/**/*.mp3",
        "/kaggle/working/ComfyUI/output/**/*.wav",
        "/kaggle/working/ComfyUI/output/**/*.flac",
        "/kaggle/working/ComfyUI/output/*.mp3",
        "/kaggle/working/ComfyUI/output/*.wav",
        "/kaggle/working/ComfyUI/output/*.flac",
        "/kaggle/working/*.mp3",
        "/kaggle/working/*.wav",
        "/kaggle/working/*.flac",
    ]
    all_audio = []
    for p in search_paths:
        all_audio.extend(glob.glob(p, recursive=True))
    all_audio = sorted(set(all_audio), key=os.path.getmtime, reverse=True)

    if not all_audio:
        display(HTML(
            "<div style='background:#fff3cd;padding:15px;border-radius:8px;"
            "border-left:5px solid #ffc107;'><b>⚠️ No audio files found.</b> "
            "Run Block 4 first.</div>"
        ))
        return

    print("📦 Found {} track(s):".format(len(all_audio)))
    total = 0.0
    for t in all_audio:
        mb = round(os.path.getsize(t) / (1024 * 1024), 2)
        total += mb
        ct = time.strftime('%H:%M:%S', time.localtime(os.path.getmtime(t)))
        print("  • {:<48} {:>6.2f} MB  [{}]".format(os.path.basename(t), mb, ct))
    print("  " + "-" * 45)
    print("  TOTAL: {:.2f} MB".format(total))
    print()

    zip_path = "/kaggle/working/YuE2_All_Tracks.zip"
    if os.path.exists(zip_path):
        os.remove(zip_path)

    print("📦 Packing archive...")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in all_audio:
            zf.write(f, arcname=os.path.basename(f))

    size_mb = round(os.path.getsize(zip_path) / (1024 * 1024), 2)
    print("✅ Archive ready: {:.2f} MB at {}".format(size_mb, zip_path))
    print()

    SAFE_LIMIT_MB = 80
    auto_triggered = False

    if size_mb <= SAFE_LIMIT_MB:
        print("🚀 Triggering auto-download via JS Blob...")
        try:
            with open(zip_path, "rb") as f:
                raw = f.read()
            b64 = base64.b64encode(raw).decode("ascii")

            js = """
            (function() {
                try {
                    var b64 = "%s";
                    var bin = atob(b64);
                    var len = bin.length;
                    var bytes = new Uint8Array(len);
                    for (var i = 0; i < len; i++) bytes[i] = bin.charCodeAt(i);
                    var blob = new Blob([bytes], {type: 'application/zip'});
                    var url = URL.createObjectURL(blob);
                    var a = document.createElement('a');
                    a.href = url;
                    a.download = 'YuE2_All_Tracks.zip';
                    document.body.appendChild(a);
                    a.click();
                    setTimeout(function() {
                        document.body.removeChild(a);
                        URL.revokeObjectURL(url);
                    }, 200);
                    console.log('[YuE2] Auto-download triggered.');
                } catch(e) {
                    console.error('[YuE2] Blob failed:', e);
                }
            })();
            """ % b64

            display(Javascript(js))
            auto_triggered = True
            print("✅ Auto-download sent to browser. Check Downloads tab.")
        except Exception as e:
            print("⚠️ Blob method error: {}".format(e))
    else:
        print("ℹ️ Archive {:.2f} MB > {} MB limit. Skipping auto-download.".format(size_mb, SAFE_LIMIT_MB))

    print()
    if auto_triggered:
        note = "If the auto-download didn't start (popup blocker?), use the link or panel below."
    else:
        note = "Use the link or the Output panel below to download manually."

    display(HTML(
        "<div style='background:#e8f5e9;padding:16px;border-radius:10px;"
        "border-left:6px solid #4caf50;margin:12px 0;font-family:sans-serif;'>"
        "<h3 style='margin:0 0 10px 0;color:#2e7d32;'>⬇️ Download Options</h3>"
        "<p style='margin:4px 0;font-size:14px;'>" + note + "</p>"
        "<p style='margin:6px 0;'><b>Option A:</b> click the blue link below.</p>"
        "<p style='margin:6px 0;'><b>Option B (100% works):</b> right sidebar → "
        "<b>Output</b> tab → <code>YuE2_All_Tracks.zip</code> → ⬇️ arrow.</p>"
        "<p style='margin:6px 0;font-size:12px;color:#666;'>"
        "Archive: <b>YuE2_All_Tracks.zip</b> • {:.2f} MB • {} tracks</p>"
        "</div>".format(size_mb, len(all_audio))
    ))

    display(FileLink(zip_path))

    if len(all_audio) <= 20:
        display(HTML(
            "<div style='background:#f5f5f5;padding:10px;border-radius:8px;"
            "margin:10px 0;'><b>🎵 Individual track links:</b></div>"
        ))
        for t in all_audio:
            mb = round(os.path.getsize(t) / (1024 * 1024), 2)
            print("  • {} ({:.2f} MB)".format(os.path.basename(t), mb))
            display(FileLink(t))

smart_download_tracks()